# Tutorial 3: Cross-Platform Comparison

SAHA profiled the same organs on more than one spatial platform. This tutorial shows how to:

1. Find matched organ samples across platforms via metadata
2. Open the SpatialData objects and harmonize their tables
3. Compare cell type composition and gene detection
4. Compute shared gene expression correlations

The v1 release covers **CosMx and Xenium**. Other platforms appear in the
`platform` enum for future releases but ship no data in v1.


In [ ]:
# !pip install scanpy anndata zarr s3fs fsspec matplotlib seaborn scipy

In [ ]:
import pandas as pd
import numpy as np
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr

sc.settings.verbosity = 1
sc.settings.set_figure_params(dpi=100, frameon=False)

import os

# One setting picks the data source; every path below is built from DATA_ROOT.
#   "s3://saha-public"     reviewer copy on Wasabi: public read, no credentials (default)
#   "s3://saha-open-data"  AWS Open Data bucket, once it is live
#   "/path/to/saha"        a local copy with the same layout: metadata/, processed/<record>/
DATA_ROOT = os.environ.get("SAHA_DATA_ROOT", "s3://saha-public")
if DATA_ROOT.startswith("s3://saha-public"):
    S3_OPTS = {"anon": True, "client_kwargs": {"endpoint_url": "https://s3.wasabisys.com"}}
elif DATA_ROOT.startswith("s3://"):
    S3_OPTS = {"anon": True}
else:
    S3_OPTS = {}
META = f"{DATA_ROOT}/metadata"

import fsspec, zarr
import anndata as ad

def object_path(sample_id):
    """The store for one acquisition: processed/<record>/<sample_id>.zarr.
    (25 Xenium stores carry an _RNA suffix, so both names are accepted.)"""
    fs, root = fsspec.core.url_to_fs(DATA_ROOT, **S3_OPTS)
    hits = [h for h in fs.glob(f"{root}/processed/*/{sample_id}*.zarr")
            if h.rsplit("/", 1)[-1] in (f"{sample_id}.zarr", f"{sample_id}_RNA.zarr")]
    assert len(hits) == 1, f"{sample_id}: {hits}"
    return fs.unstrip_protocol(hits[0]) if DATA_ROOT.startswith("s3://") else hits[0]

def read_table(sample_id):
    """The cell table (an AnnData) of one object, read straight from the store.
    Only the table is fetched; images, polygons and transcripts stay where they are."""
    g = zarr.open_group(object_path(sample_id), mode="r", storage_options=S3_OPTS or None)
    adata = ad.io.read_elem(g["tables/table"])
    # 25 Xenium tables keep scaled values in X and the counts in layers["raw"];
    # put counts in X so every object starts from the same place.
    if "raw" in adata.layers:
        adata.X = adata.layers["raw"]
    return adata

# Cell types: `cell_type_broad` is the one vocabulary shared by every object.
LABEL = "cell_type_broad"

# For the full SpatialData object (images, polygons, transcripts), copy the store
# first; spatialdata does not open a remote store reliably. For example:
#   aws s3 sync --no-sign-request --endpoint-url https://s3.wasabisys.com \
#       s3://saha-public/processed/<record>/<store>.zarr ./<store>.zarr
#   sdata = spatialdata.read_zarr("./<store>.zarr")


## Step 1: Find matched samples across platforms

In [ ]:
samples = pd.read_parquet(f"{META}/samples.parquet", storage_options=S3_OPTS or None)
donors  = pd.read_parquet(f"{META}/donors.parquet",  storage_options=S3_OPTS or None)

# Organs profiled by multiple platforms (RNA only)
multi_platform = (
    samples[samples["assay_type"] == "rna"]
    .groupby("organ")["platform"]
    .nunique()
)
print("Organs with multi-platform coverage:")
print(multi_platform[multi_platform > 1])

In [ ]:
# Ileum: the ILE_A1 tissue microarray was imaged on both CosMx and Xenium
# (different sections of the same block). Colon has no Xenium in this release.
TARGET_ORGAN = "ileum"
TARGET_SLIDE = "ILE_A1"

pair = samples[
    (samples["slide_id"] == TARGET_SLIDE) &
    (samples["assay_type"] == "rna") &
    (samples["platform"].isin(["cosmx", "xenium"]))
].copy()

print(pair.groupby("platform")["n_cells"].describe())

In [ ]:
# One sample per platform; the smallest keeps the download small
cosmx_row  = pair[pair["platform"] == "cosmx" ].nsmallest(1, "n_cells").iloc[0]
xenium_row = pair[pair["platform"] == "xenium"].nsmallest(1, "n_cells").iloc[0]

print(f"CosMx sample:  {cosmx_row['sample_id']}  ({cosmx_row['n_cells']:,} cells)")
print(f"Xenium sample: {xenium_row['sample_id']} ({xenium_row['n_cells']:,} cells)")

## Step 2: Load both cell tables

In [ ]:
adata_cosmx  = read_table(cosmx_row["sample_id"])
adata_xenium = read_table(xenium_row["sample_id"])

# Tag each with platform
adata_cosmx.obs["platform"]  = "cosmx"
adata_xenium.obs["platform"] = "xenium"

print(f"CosMx:  {adata_cosmx.n_obs:,} cells x {adata_cosmx.n_vars:,} genes")
print(f"Xenium: {adata_xenium.n_obs:,} cells x {adata_xenium.n_vars:,} genes")

## Step 3: Shared gene overlap

In [ ]:
shared_genes = sorted(set(adata_cosmx.var_names) & set(adata_xenium.var_names))
print(f"CosMx genes:   {adata_cosmx.n_vars:,}")
print(f"Xenium genes:  {adata_xenium.n_vars:,}")
print(f"Shared:        {len(shared_genes):,} ({100*len(shared_genes)/adata_xenium.n_vars:.0f}% of Xenium panel)")

# Subset to shared genes
adata_c = adata_cosmx[:, shared_genes].copy()
adata_x = adata_xenium[:, shared_genes].copy()
sc.pp.filter_cells(adata_c, min_counts=10)
sc.pp.filter_cells(adata_x, min_counts=10)

## Step 4: Cell type composition comparison

In [ ]:
def cell_type_fractions(adata, label):
    frac = adata.obs[LABEL].value_counts(normalize=True).rename(label)
    return frac

frac_c = cell_type_fractions(adata_cosmx,  "CosMx")
frac_x = cell_type_fractions(adata_xenium, "Xenium")

comp = pd.concat([frac_c, frac_x], axis=1).fillna(0)
comp = comp.sort_values("CosMx", ascending=False)

fig, ax = plt.subplots(figsize=(9, 5))
comp.plot(kind="bar", ax=ax, width=0.7)
ax.set_ylabel("Cell fraction")
ax.set_title(f"Cell type composition — {TARGET_ORGAN}")
ax.set_xticklabels(ax.get_xticklabels(), rotation=40, ha="right", fontsize=9)
plt.tight_layout()
plt.show()

## Step 5: Per-cell-type gene expression correlation

In [ ]:
def mean_expression(adata, cell_type_col=LABEL):
    """Return (cell_types × genes) mean log-normalized expression."""
    adata = adata.copy()
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)
    return (
        pd.DataFrame(
            adata.X.toarray() if hasattr(adata.X, "toarray") else adata.X,
            index=adata.obs_names,
            columns=adata.var_names,
        )
        .join(adata.obs[[cell_type_col]])
        .groupby(cell_type_col)
        .mean()
    )

mean_c = mean_expression(adata_c)
mean_x = mean_expression(adata_x)

shared_ct = list(set(mean_c.index) & set(mean_x.index))
print(f"Shared cell types: {shared_ct}")

In [ ]:
# Spearman correlation per cell type across shared genes
corr_results = []
for ct in shared_ct:
    r, p = spearmanr(mean_c.loc[ct], mean_x.loc[ct])
    corr_results.append({"cell_type": ct, "spearman_r": r, "p_value": p})

corr_df = pd.DataFrame(corr_results).sort_values("spearman_r", ascending=False)

fig, ax = plt.subplots(figsize=(7, 4))
ax.barh(corr_df["cell_type"], corr_df["spearman_r"], color="mediumseagreen")
ax.axvline(0.9, color="gray", linestyle="--", lw=0.8, label="r = 0.9")
ax.set_xlabel("Spearman r (CosMx vs Xenium mean expression)")
ax.set_title(f"Cross-platform expression concordance — {TARGET_ORGAN}")
ax.legend()
plt.tight_layout()
plt.show()

print(corr_df.to_string(index=False))

## Step 6: Scatter plot for one cell type

In [ ]:
FOCUS_CT = corr_df.iloc[0]["cell_type"]  # highest correlation cell type

x_vals = mean_c.loc[FOCUS_CT]
y_vals = mean_x.loc[FOCUS_CT]

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(x_vals, y_vals, s=3, alpha=0.5, color="steelblue")

# Annotate top 10 highest-expressed genes
top_genes = x_vals.nlargest(10).index
for g in top_genes:
    ax.annotate(g, (x_vals[g], y_vals[g]), fontsize=7,
                xytext=(3, 3), textcoords="offset points")

ax.set_xlabel("CosMx log1p mean expression")
ax.set_ylabel("Xenium log1p mean expression")
ax.set_title(f"{FOCUS_CT} — CosMx vs Xenium (r={corr_df.iloc[0]['spearman_r']:.3f})")

# identity line
lim = max(ax.get_xlim()[1], ax.get_ylim()[1])
ax.plot([0, lim], [0, lim], "k--", lw=0.8, alpha=0.4)

plt.tight_layout()
plt.show()

## Step 7: Concatenate for joint UMAP

In [ ]:
# Concatenate (shared genes only)
combined = sc.concat(
    {"cosmx": adata_c, "xenium": adata_x},
    label="platform",
    uns_merge="unique",
)

# Normalize + PCA
sc.pp.normalize_total(combined, target_sum=1e4)
sc.pp.log1p(combined)
# The shared panel is small (~125 genes), so every shared gene goes into the PCA
sc.pp.pca(combined, n_comps=30)

# Optionally apply batch correction (scVI, Harmony, etc.)
# Here we just use raw PCA for illustration
sc.pp.neighbors(combined, n_neighbors=15)
sc.tl.umap(combined)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sc.pl.umap(combined, color="platform",   ax=axes[0], show=False, frameon=False,
           title="Colored by platform")
sc.pl.umap(combined, color=LABEL,  ax=axes[1], show=False, frameon=False,
           legend_loc="on data", legend_fontsize=7,
           title="Colored by cell type")
plt.tight_layout()
plt.show()

## Next steps

- Apply `scvi-tools` or `harmonypy` for principled batch correction across platforms
- Compare matched RNA and protein on the same tissue — CosMx runs both, and the
  `assay_type` column in `samples.parquet` distinguishes them
- See [QUICK_START.md](../docs/QUICK_START.md) for other access patterns
